# Statistik — Job Market Indonesia
Dataset kecil & riil: uji dilakukan dengan sadar batasannya — semua kesimpulan disertai keterbatasan, bukan overclaim.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
lv = pd.read_csv('../data/processed/lowongan.csv')
sk = pd.read_csv('../data/processed/job_skill.csv')

## 1. Apakah judul 'analyst' berhubungan dengan senioritas?
Chi-square: `peran_analyst` × `senioritas`

In [2]:
ct = pd.crosstab(lv.peran_analyst, lv.senioritas)
chi2, p, dof, _ = stats.chi2_contingency(ct)
n_ = ct.values.sum()
v = np.sqrt(chi2 / (n_ * (min(ct.shape) - 1)))
print(ct)
print(f'chi2={chi2:.1f}, p={p:.4f}, dof={dof}, CramérV={v:.3f}')
print('p<0.05:', p < 0.05)

senioritas     Junior / Entry  Mid  Senior / Lead
peran_analyst                                    
False                      51  720            263
True                       20  355            115
chi2=1.4, p=0.4945, dof=2, CramérV=0.030
p<0.05: False


## 2. Apakah lowongan analyst lebih terbuka ke fresh grad?
Chi-square: `peran_analyst` × `fresh_grad_ok`

In [3]:
ct2 = pd.crosstab(lv.peran_analyst, lv.fresh_grad_ok)
chi2, p, dof, _ = stats.chi2_contingency(ct2)
rate = lv.groupby('peran_analyst').fresh_grad_ok.mean()
print(ct2, '\n', rate.round(3))
print(f'chi2={chi2:.2f}, p={p:.4f}')

fresh_grad_ok  False  True 
peran_analyst              
False            924    110
True             442     48 
 peran_analyst
False    0.106
True     0.098
Name: fresh_grad_ok, dtype: float64
chi2=0.17, p=0.6789


## 3. Gaji: Junior vs Senior
⚠️ Hanya ~48 lowongan menampilkan gaji (~3%) → power rendah. Hasil bersifat indikatif, BUKAN estimasi pasar.

In [4]:
bg = lv.dropna(subset=['gaji_max_jt'])
print(bg.groupby('senioritas').gaji_max_jt
        .agg(['count','median','mean']).round(1))
jun = bg[bg.senioritas=='Junior / Entry'].gaji_max_jt
sen = bg[bg.senioritas=='Senior / Lead'].gaji_max_jt
if len(jun) >= 3 and len(sen) >= 3:
    u, p = stats.mannwhitneyu(jun, sen, alternative='less')
    print(f'Mann-Whitney (junior < senior): U={u:.0f}, p={p:.4f}')
else:
    print('n per grup terlalu kecil untuk uji — dilaporkan deskriptif saja')

                count  median  mean
senioritas                         
Junior / Entry      3     6.8   6.8
Mid                33     9.0  11.4
Senior / Lead      12    17.5  18.6
Mann-Whitney (junior < senior): U=4, p=0.0210


## 4. Apakah domain menentukan peluang fresh-grad?
Chi-square: `domain` × `fresh_grad_ok` — domain dengan expected count <5 digabung 'Lainnya'.

In [5]:
dom = lv.domain.value_counts()
lv3 = lv.copy()
lv3['dom'] = np.where(lv3.domain.isin(dom[dom>=50].index),
                      lv3.domain, 'Lainnya')
ctd = pd.crosstab(lv3.dom, lv3.fresh_grad_ok)
chi2, p, dof, exp = stats.chi2_contingency(ctd)
print(ctd, '\n')
print((ctd.div(ctd.sum(1), axis=0)[True]*100)
      .round(1).rename('fresh-grad %'))
print(f'chi2={chi2:.1f}, p={p:.4g}, dof={dof}')

fresh_grad_ok         False  True 
dom                               
Adjacent (non-IT)       179     72
Data & Analytics        347     36
IT Mgmt & Consulting    175     14
IT Support               52      3
Infra & Network         119      2
Lainnya                  30      2
Security                 49      3
Software Engineering    415     26 

dom
Adjacent (non-IT)       28.7
Data & Analytics         9.4
IT Mgmt & Consulting     7.4
IT Support               5.5
Infra & Network          1.7
Lainnya                  6.2
Security                 5.8
Software Engineering     5.9
Name: fresh-grad %, dtype: float64
chi2=115.4, p=7.027e-22, dof=7


C:\Users\PKL01\AppData\Local\Temp\ipykernel_5096\1474386840.py:8: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  print((ctd.div(ctd.sum(1), axis=0)[True]*100)


## 5. Platform × domain — dua platform, dua pasar
Apakah komposisi domain berbeda antara Kalibrr dan LinkedIn?

In [6]:
ctp = pd.crosstab(lv3.dom, lv3.sumber)
chi2, p, dof, _ = stats.chi2_contingency(ctp)
n_ = ctp.values.sum()
v = np.sqrt(chi2 / (n_ * (min(ctp.shape) - 1)))
print(ctp)
print(f'chi2={chi2:.1f}, p={p:.4g}, CramérV={v:.3f}')
print('CramérV > 0.3 = asosiasi kuat antar platform dan domain')

sumber                kalibrr  linkedin
dom                                    
Adjacent (non-IT)         177        74
Data & Analytics           55       328
IT Mgmt & Consulting       31       158
IT Support                 10        45
Infra & Network            16       105
Lainnya                     1        31
Security                   11        41
Software Engineering       77       364
chi2=342.0, p=6.469e-70, CramérV=0.474
CramérV > 0.3 = asosiasi kuat antar platform dan domain


## 6. Skill × keluarga peran — mana yang membedakan?
Untuk tiap skill top-8: chi-square hadir/tidak × role_family (family kecil digabung 'Lainnya').

In [7]:
lv2 = lv.copy()
top_fam = lv2.role_family.value_counts()
lv2['fam'] = np.where(lv2.role_family.isin(top_fam[top_fam>=20].index),
                      lv2.role_family, 'Lainnya')
hasil = []
for s in sk.skill.value_counts().head(8).index:
    ids = set(sk[sk.skill == s].id_lowongan)
    lv2['ada'] = lv2.id_lowongan.isin(ids)
    ct = pd.crosstab(lv2.fam, lv2.ada)
    if ct.shape == (len(lv2.fam.unique()), 2):
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        hasil.append((s, round(chi2,1), round(p,4)))
pd.DataFrame(hasil, columns=['skill','chi2','p']).sort_values('p')

,skill,chi2,p
0,SQL,308.8,0.0
1,Python,312.5,0.0
2,API / Integrasi,459.9,0.0
3,Networking,275.7,0.0
4,Excel,361.1,0.0
5,Git,323.4,0.0
6,JavaScript/TypeScript,530.4,0.0
7,Cloud (AWS/GCP/Azure),232.0,0.0


## Kesimpulan
- Uji signifikan → laporkan; tidak signifikan → katakan jujur.
- Dataset riil ~1.500 lowongan: uji bermakna untuk efek sedang-besar; efek sangat kecil tetap sulit dideteksi.
- Analisis gaji dibatasi ~3% coverage: kesimpulan gaji bersifat hipotesis, bukan estimasi pasar.
- Hasil non-signifikan dilaporkan apa adanya — bukan dirapikan demi narasi.